In [0]:
from delta.tables import DeltaTable

In [0]:
df_bronze=spark.read.table("fintrack.bronze.transaction")
df_bronze.count()

1150

In [0]:
import pyspark.sql.functions as f

silver_trans=spark.read.table("fintrack.silver.transaction")
silver_trans.count()

1022

In [0]:
#  silver_trans=spark.read.table("fintrack.silver.transaction")
#  silver_trans.count()
 


In [0]:
df_bronze = df_bronze.dropDuplicates(["txn_id"])
print(f"After drop duplicates  : {df_bronze.count()}")

After drop duplicates  : 1100


In [0]:
main_table = DeltaTable.forName(spark, "fintrack.silver.transaction")
main_table.alias("t") \
    .merge(df_bronze.alias("l"), "t.txn_id = l.txn_id") \
    .whenMatchedUpdate(set={"status": "l.status"}) \
    .whenNotMatchedInsertAll() \
    .execute() 
main_table.toDF().count()

1100

In [0]:
df_silver=spark.read.table("fintrack.silver.transaction")


In [0]:
%sql
update fintrack.silver.transaction
Set Status="FRAUD_REVIEW"
where is_fraud=true;

num_affected_rows
47


In [0]:
%sql
delete from fintrack.silver.transaction
where status="FAILED" AND amount <100 ;

num_affected_rows
78


In [0]:
%sql
select count(*) as total_count,
count (case when status="FAILED" AND amount < 100 then 1 end) as failed_count,
(total_count-failed_count) as current_count
from fintrack.silver.transaction;

total_count,failed_count,current_count
1022,0,1022


In [0]:

spark.sql("SHOW TABLES IN fintrack.Silver").display()

database,tableName,isTemporary
silver,complete_transaction,false
silver,late_transaction,false
silver,transaction,false


In [0]:

%sql
optimize fintrack.silver.transaction;
vacuum fintrack.silver.transaction retain 168 hours;



path
""


In [0]:

spark.sql("describe history fintrack.silver.late_transaction").display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
20,2026-03-30T07:14:52.000Z,74832140431509,nirendraprabhu750@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3693218179253919),e9ea3b5d-412d-4900-af27-94b9316f04a6,0330-062513-wj53b1c7-v2n,19,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4459, numDeletionVectorsRemoved -> 0, numOutputRows -> 150, numOutputBytes -> 4459)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
19,2026-03-30T06:40:12.000Z,74832140431509,nirendraprabhu750@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3693218179253919),f20e3a69-c8ba-4871-80e5-10398ccfbbdb,0330-062513-wj53b1c7-v2n,18,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4459, numDeletionVectorsRemoved -> 0, numOutputRows -> 150, numOutputBytes -> 4459)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
18,2026-03-30T05:57:32.000Z,74832140431509,nirendraprabhu750@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3693218179253919),bdf69e13-9ba3-42c1-bf28-7a17d9ac0159,0330-055258-g4y08hfg-v2n,17,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4459, numDeletionVectorsRemoved -> 0, numOutputRows -> 150, numOutputBytes -> 4459)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
17,2026-03-30T05:54:28.000Z,74832140431509,nirendraprabhu750@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3693218179253919),b3e25f65-54b2-4ec1-a245-3db6575c8f44,0330-055258-g4y08hfg-v2n,16,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4459, numDeletionVectorsRemoved -> 0, numOutputRows -> 150, numOutputBytes -> 4459)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
16,2026-03-27T11:33:02.000Z,74832140431509,nirendraprabhu750@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3693218179253919),67b7a6fc-2a03-4623-b4a0-4e8b26b14a7f,0327-111521-5jv9g0qp-v2n,15,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4459, numDeletionVectorsRemoved -> 0, numOutputRows -> 150, numOutputBytes -> 4459)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
15,2026-03-27T10:33:54.000Z,74832140431509,nirendraprabhu750@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3693218179253919),9aebd7bd-dad8-4e4f-b859-6acca3e9f559,0327-085850-m0dj8xx7-v2n,14,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4459, numDeletionVectorsRemoved -> 0, numOutputRows -> 150, numOutputBytes -> 4459)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
14,2026-03-27T10:22:16.000Z

In [0]:
spark.sql("select * from fintrack.silver.transaction").count()

1022